In [0]:
%run "../notebooks/helper_functions"

#### Source table configuration
This cell defines which source tables will be loaded into Bronze and which columns should be used as:
- **primary key**
- **timestamp / watermark colum**
It also creates a unique **bronze_run_id** for the current pipeline run.

In [0]:
import uuid

In [0]:
tables_config = {
  "orders" : {"pk_col": "order_id", "ts_col": "updated_at"},
  "products" : {"pk_col": "product_id", "ts_col": "updated_at"},
  "payments" : {"pk_col": "payment_id", "ts_col": "processed_at"},
}

bronze_run_id = str(uuid.uuid4())

print(f"Current Bronze Run ID is: {bronze_run_id}")

#### Bronze incremental load loop
This is the main Bronze logic
For each table, the notebook:
1. reads the last watermark
2. reads the source SQL table
3. filters only new/changed rows
4. adds Bronze audit columns
5. appends the rows into the Bronze Delta table
6. update the control table

This is the core incremental loading logic

In [0]:
for table_name, cfg in tables_config.items():
  pk_col = cfg.get("pk_col")
  ts_col = cfg.get("ts_col")
  source_table = f"novacart_external_mssql_oltp_db.dbo.{table_name}"
  target_table = f"novacart_catalog.bronze_schema.{table_name}_raw"
  last_successful_ts,last_successful_pk = get_last_successfull_watermark(table_name)
  print(f"\n*** Processing {table_name} ***")
  print(f"Last successful ts: {last_successful_ts}")
  print(f"Last successful pk: {last_successful_pk}")

  source_df = spark.read.table(source_table) \
    .withColumn(ts_col, F.col(ts_col).cast("timestamp")) \
  
  if last_successful_ts is None:
    rows_to_load_df = source_df
  else:
    rows_to_load_df = source_df.filter(
      (F.col(ts_col) > F.lit(last_successful_ts)) |
      (
        (F.col(ts_col) == F.lit(last_successful_ts)) &
        (F.col(pk_col).cast("long") > F.lit("last_successful_pk"))
      )
    )


  rows_to_load_df = (
    rows_to_load_df
    .withColumn("bronze_ingested_at", F.current_timestamp())
    .withColumn("bronze_run_id", F.lit(bronze_run_id))
    .withColumn("bronze_source_table", F.lit(source_table))
  )

  rows_count = rows_to_load_df.count()
  print(f"{table_name} rows_to_load = {rows_count}")

  if rows_count == 0:
    print(f"No new rows to load for {table_name}.")
    upsert_bronze_control(
      table_name,
      ts_col,
      pk_col,
      last_successful_ts,
      last_successful_pk,
      rows_count,
      bronze_run_id
    )
    continue
  
  rows_to_load_df.write.format("delta").mode("append").saveAsTable(target_table)

  max_ts = rows_to_load_df.agg(F.max(ts_col).alias("max_ts")).collect()[0]["max_ts"]


  max_pk = (
    rows_to_load_df
    .filter(F.col(ts_col) == F.lit(max_ts))
    .agg(F.max(pk_col).cast("long").alias("max_pk"))
    .collect()[0]["max_pk"]
  )

  upsert_bronze_control(
      table_name,
      ts_col,
      pk_col,
      max_ts,
      max_pk,
      rows_count,
      bronze_run_id
    )
  
  print(f"Wrote {rows_count} to {target_table}")
  
  

#### Quick validation

In [0]:
print("Orders Bronze count:", spark.sql("select count(*) from novacart_catalog.bronze_schema.orders_raw"))

print("Products Bronze count:", spark.sql("select count(*) from novacart_catalog.bronze_schema.products_raw"))

print("Payments Bronze count:", spark.sql("select count(*) from novacart_catalog.bronze_schema.payments_raw"))


display(spark.sql("select * from novacart_catalog.bronze_schema.ingestion_control").orderBy("table_name"))